# CS-781 Capstone, Phase 2: Legal NER with the Mamba-2 / attention hybrid (Phase-1 model)

One notebook, three experiments. **The backbone, head, windowing, decoding and training budget are identical; only the adaptation technique changes**, so the runs are a controlled comparison and count as three methodologically different submissions.

| `EXPERIMENT` | Technique | Trainable parts |
|---|---|---|
| `M1_full` | Full fine-tuning | every backbone weight + head |
| `M2_lora` | LoRA (r=16) on every linear layer | LoRA matrices + head, backbone frozen |
| `M3_adapter` | Bottleneck adapters (dim 64) after every layer | adapters + head, backbone frozen |

**Pipeline:** load Phase-1 checkpoint -> audit data and offset convention -> normalise text exactly like Phase 1 (with a normalised-to-raw index map) -> BPE tokenise with character offsets -> BIO labels -> overlapping windows -> Mamba hybrid (+ BiGRU for right context) -> train on fold1, select epoch on fold2 (strict F1) -> retrain on fold1+fold2 -> predict fold3 -> convert back to **raw character offsets** -> validate -> `submission.zip`.

**How to run:** set the paths in section 1, pick `EXPERIMENT`, run all cells. Repeat with the other two values (each run writes to its own folder, and the last cell compares all runs found).

**Why a BiGRU on top:** the Phase-1 Mamba hybrid is causal (left context only), but tagging an entity often needs the words after it. A small bidirectional GRU over the backbone's hidden states supplies right context. It is part of the head, trained in all three experiments.

**Why overlapping windows with averaged scores:** the model's attention layers were trained with 1280-token context, but judgments are longer. Documents are cut into 1024-token windows with overlap; per-token log-probabilities are averaged (edge tokens get less weight), then decoded once per document so entities are never stitched across windows.

## 1. Configuration

In [1]:
import os, sys, json, re, math, time, random, zipfile, heapq, shutil, subprocess, hashlib, bisect
from collections import Counter, defaultdict
from dataclasses import dataclass
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint

DEBUG = False                 # True = tiny synthetic smoke test
SEED = 42
EXPERIMENT = "M3_adapter"        # M1_full | M2_lora | M3_adapter

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
# ---- Phase-1 assets of the Mamba hybrid (your uploaded Kaggle dataset) ----
CKPT_PATH = "/kaggle/input/models/ashishkrjha227/mamba-implementation/pytorch/default/1/best.pt"                # hybrid checkpoint (dict with "model" and "cfg")
TOK_PATH  = "/kaggle/input/models/ashishkrjha227/mamba-implementation/pytorch/default/1/bpe_merges.json"        # the tokenizer THIS checkpoint was trained with
OUTPUT_DIR = "/kaggle/working/phase2_lner"
EOS_PLACEHOLDER = "<|endoftext|>"

PRESETS = {
    "M1_full":    dict(method="full",    lr_backbone=1e-4, lr_head=1e-3,
                       desc="Full fine-tuning of all Mamba-hybrid weights"),
    "M2_lora":    dict(method="lora",    lr_backbone=1e-3, lr_head=1e-3, lora_r=16, lora_alpha=32, lora_dropout=0.05,
                       desc="LoRA r=16 on every linear layer, backbone frozen"),
    "M3_adapter": dict(method="adapter", lr_backbone=1e-3, lr_head=1e-3, adapter_dim=64,
                       desc="Bottleneck adapters after every layer, backbone frozen"),
}

# ---- shared by all three experiments (the controlled part) ----
DEV_FOLD = "fold2"            # train on fold1, select the epoch on fold2; the final model trains on both
WINDOW, OVERLAP, TRAIN_STRIDE = 1024, 192, 512     # tokens (must be <= the model's 1280 context)
MICRO_BS, GRAD_ACCUM = 4, 4
EPOCHS = 10
WARMUP_FRAC, WEIGHT_DECAY, CLIP = 0.06, 0.01, 1.0
CONTEXT_HEAD = "bigru"        # "bigru" (right context from a small BiGRU) | "none" (linear head on causal states)
GRU_HIDDEN, HEAD_DROPOUT = 384, 0.1
CLASS_WEIGHT = "sqrt_inv"     # "sqrt_inv" (rare entity tags up-weighted, capped) | "none"
NEG_WINDOW_KEEP = 0.5         # fraction of entity-free training windows kept each epoch
DECODE_FOR_SELECTION = "viterbi"      # "viterbi" (BIO-constrained) | "argmax"
GRAD_CKPT = False             # True = less memory, ~30% slower
FINAL_RETRAIN = True          # retrain on fold1+fold2 for the best epoch count, then predict fold3
MAX_TRAIN_MINUTES = 100       # per training phase; stops early and keeps the best epoch so far
INCLUDE_ENTITY_TEXT = False   # optional 4th field in prediction.json (ignored by the scorer)

if DEBUG:
    WINDOW, OVERLAP, TRAIN_STRIDE = 96, 24, 48
    MICRO_BS, GRAD_ACCUM, EPOCHS, GRU_HIDDEN = 4, 1, 3, 32
    MAX_TRAIN_MINUTES = 10

EXP = PRESETS[EXPERIMENT]
EXP_DIR = f"{OUTPUT_DIR}/{EXPERIMENT}"
os.makedirs(EXP_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "", "| experiment:", EXPERIMENT, "-", EXP["desc"])

device: cuda | Tesla T4 | experiment: M3_adapter - Bottleneck adapters after every layer, backbone frozen


## 2. Phase-1 code, reused unchanged
`normalize_text`, the from-scratch `ByteBPE` tokenizer and the Mamba-2 / attention hybrid, copied from the Phase-1 notebooks so the checkpoint loads exactly.

In [2]:
CHAR_MAP = str.maketrans({
    "\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"', "\u2013": "-", "\u2014": "-",
    "\u2212": "-", "\u2026": "...", "\u00a0": " ", "\u200b": "", "\ufeff": "", "\u00ad": "",
})
CTRL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")

def normalize_text(t: str, keep_newlines: bool) -> str:
    """Character and whitespace normalisation. Applied to train text AND test prompts."""
    t = CTRL.sub(" ", t.translate(CHAR_MAP))
    if keep_newlines:
        t = re.sub(r"[ \t]+", " ", t)
        t = re.sub(r" ?\n ?", "\n", t)
        t = re.sub(r"\n{3,}", "\n\n", t)
    else:
        t = re.sub(r"\s+", " ", t)
    return t.strip()

PRETOKEN_RE = re.compile(r"""'(?:[sdmt]|ll|ve|re)| ?[^\W\d_]+| ?\d{1,3}| ?(?:[^\s\w]|_)+|\s+(?!\S)|\s+""")

class ByteBPE:
    def __init__(self, merges=None):
        self.merges = [tuple(m) for m in (merges or [])]
        self.rank = {p: i for i, p in enumerate(self.merges)}
        self.vocab = [bytes([i]) for i in range(256)]
        for a, b in self.merges:
            self.vocab.append(self.vocab[a] + self.vocab[b])
        self.eos_id = len(self.vocab)
        self.vocab.append(EOS_PLACEHOLDER.encode())
        self.vocab_size = len(self.vocab)
        self._cache = {}

    # ---------- training ----------
    @staticmethod
    def train(chunk_counts, vocab_size, min_freq=3, log_every=2000):
        n_merges = vocab_size - 256 - 1                      # last id is EOS
        items = [(c.encode("utf-8"), f) for c, f in chunk_counts.items() if f >= min_freq]
        words = [list(b) for b, _ in items]
        freqs = [f for _, f in items]
        pair_cnt, where = defaultdict(int), defaultdict(set)
        for i, w in enumerate(words):
            for p in zip(w, w[1:]):
                pair_cnt[p] += freqs[i]; where[p].add(i)
        heap = [(-c, p) for p, c in pair_cnt.items()]
        heapq.heapify(heap)
        merges, t0 = [], time.time()
        while len(merges) < n_merges:
            best = None
            while heap:                                       # lazy deletion
                negc, p = heapq.heappop(heap)
                c = pair_cnt.get(p, 0)
                if c <= 0: continue
                if c != -negc:
                    heapq.heappush(heap, (-c, p)); continue
                best = p; break
            if best is None: break
            a, b = best; new_id = 256 + len(merges); merges.append(best)
            changed = set()
            for i in list(where[best]):
                w, f = words[i], freqs[i]
                out, j, n, hit = [], 0, len(w), False
                while j < n:
                    if j < n - 1 and w[j] == a and w[j + 1] == b:
                        out.append(new_id); j += 2; hit = True
                    else:
                        out.append(w[j]); j += 1
                if not hit: continue
                for q in zip(w, w[1:]): pair_cnt[q] -= f
                for q in zip(out, out[1:]):
                    pair_cnt[q] += f; where[q].add(i); changed.add(q)
                words[i] = out
            where.pop(best, None); pair_cnt.pop(best, None)
            for q in changed:
                if pair_cnt[q] > 0: heapq.heappush(heap, (-pair_cnt[q], q))
            if len(merges) % log_every == 0:
                print(f"  merges {len(merges):6d}/{n_merges} | {time.time() - t0:5.0f}s")
        return ByteBPE(merges)

    # ---------- encoding / decoding ----------
    def _encode_chunk(self, chunk):
        hit = self._cache.get(chunk)
        if hit is not None: return hit
        ids = list(chunk.encode("utf-8"))
        while len(ids) > 1:
            best_r, best_p = None, None
            for p in zip(ids, ids[1:]):
                r = self.rank.get(p)
                if r is not None and (best_r is None or r < best_r):
                    best_r, best_p = r, p
            if best_p is None: break
            a, b = best_p; new_id = 256 + best_r; out, j = [], 0
            while j < len(ids):
                if j < len(ids) - 1 and ids[j] == a and ids[j + 1] == b:
                    out.append(new_id); j += 2
                else:
                    out.append(ids[j]); j += 1
            ids = out
        if len(self._cache) > 3_000_000: self._cache.clear()
        self._cache[chunk] = ids
        return ids

    def encode(self, text):
        ids = []
        for chunk in PRETOKEN_RE.findall(text):
            ids.extend(self._encode_chunk(chunk))
        return ids

    def decode(self, ids):
        data = b"".join(self.vocab[i] for i in ids if i != self.eos_id)
        return data.decode("utf-8", errors="replace")

    def save(self, path):
        json.dump({"merges": self.merges, "eos": EOS_PLACEHOLDER}, open(path, "w"))

    @staticmethod
    def load(path):
        return ByteBPE(json.load(open(path))["merges"])

In [3]:
@dataclass
class HybridConfig:
    vocab_size: int
    n_ctx: int = 1280
    d_model: int = 768
    pattern: str = "MMMA" * 6        # M = Mamba-2 (SSD) layer, A = attention + SwiGLU layer
    # --- Mamba-2 / SSD layer ---
    expand: int = 2                  # d_inner = expand * d_model
    headdim: int = 64                # SSM head size P  (n_ssm_heads = d_inner / headdim)
    d_state: int = 64                # state size N per head
    d_conv: int = 4                  # causal depthwise conv width
    chunk: int = 64                  # SSD chunk length Q
    # --- attention layer ---
    n_head: int = 12
    n_kv_head: int = 4
    n_ffn: int = 2048
    rope_base: float = 10000.0

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim)); self.eps = eps
    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)).type_as(x) * self.weight

class GatedRMSNorm(nn.Module):
    """Mamba-2 output norm: RMSNorm(y * silu(z))."""
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim)); self.eps = eps
    def forward(self, y, z):
        y = y * F.silu(z)
        yf = y.float()
        return (yf * torch.rsqrt(yf.pow(2).mean(-1, keepdim=True) + self.eps)).type_as(y) * self.weight

# ------------------------------------------------------------------------------------------
# State-space duality (SSD) scan, pure PyTorch. Computes, for every head h,
#     s_t = exp(dt_t * A_h) * s_{t-1} + dt_t * (x_t outer B_t)      y_t = C_t . s_t
# in chunks of Q tokens: inside a chunk it is a masked "attention-like" matmul; across chunks only a
# small recurrence on the (P x N) states is needed. Decay math is fp32, big matmuls follow autocast.
# ------------------------------------------------------------------------------------------
def ssd_chunked(X, dt, A, Bm, Cm, Q):
    """X (B,T,H,P) | dt (B,T,H) fp32 | A (H,) fp32 <0 | Bm,Cm (B,T,N). Returns Y (B,T,H,P), final state (B,H,P,N) fp32."""
    Bsz, T, H, P = X.shape
    N = Bm.size(-1)
    pad = (-T) % Q
    if pad:                                                   # dt = 0 on padding => state is untouched
        X = F.pad(X, (0, 0, 0, 0, 0, pad)); dt = F.pad(dt, (0, 0, 0, pad))
        Bm = F.pad(Bm, (0, 0, 0, pad)); Cm = F.pad(Cm, (0, 0, 0, pad))
    Tp = T + pad; nC = Tp // Q
    Xc = (X * dt.to(X.dtype)[..., None]).view(Bsz, nC, Q, H, P)             # dt-scaled input
    Bc, Cc = Bm.reshape(Bsz, nC, Q, N), Cm.reshape(Bsz, nC, Q, N)
    a_cum = (dt * A).view(Bsz, nC, Q, H).permute(0, 3, 1, 2).cumsum(-1)     # (B,H,nC,Q) fp32, non-increasing

    # 1) inside each chunk: Y = (C B^T o L) X, with L[i,j] = exp(a_i - a_j) for i >= j
    tril = torch.tril(torch.ones(Q, Q, dtype=torch.bool, device=X.device))
    L = torch.exp((a_cum[..., :, None] - a_cum[..., None, :]).masked_fill(~tril, float("-inf")))
    G = torch.einsum("bcqn,bcsn->bcqs", Cc, Bc)                              # (B,nC,Q,Q)
    M = (G[:, None].float() * L).to(Xc.dtype)                                # (B,H,nC,Q,Q)
    Y_diag = torch.einsum("bhcqs,bcshp->bcqhp", M, Xc)

    # 2) the state each chunk would produce on its own
    decay_states = torch.exp(a_cum[..., -1:] - a_cum)                        # (B,H,nC,Q)
    Xw = Xc * decay_states.permute(0, 2, 3, 1)[..., None].to(Xc.dtype)
    states = torch.einsum("bcshp,bcsn->bchpn", Xw, Bc)                       # (B,nC,H,P,N)

    # 3) pass states across chunks (tiny recurrence, fp32)
    with torch.autocast(device_type=X.device.type, enabled=False):
        states = torch.cat([torch.zeros_like(states[:, :1]), states.float()], 1)     # (B,nC+1,H,P,N)
        cs = F.pad(a_cum[..., -1], (1, 0)).cumsum(-1)                                # (B,H,nC+1)
        seg = cs[..., :, None] - cs[..., None, :]
        keep = torch.tril(torch.ones(nC + 1, nC + 1, dtype=torch.bool, device=X.device))
        decay_chunk = torch.exp(seg.masked_fill(~keep, float("-inf")))
        new_states = torch.einsum("bhzc,bchpn->bzhpn", decay_chunk, states)
    prev_states, final_state = new_states[:, :-1], new_states[:, -1]

    # 4) contribution of everything before the chunk to each token inside it
    out_decay = torch.exp(a_cum).permute(0, 2, 3, 1)[..., None].to(Xc.dtype)         # (B,nC,Q,H,1)
    Y_off = torch.einsum("bcqn,bchpn->bcqhp", Cc, prev_states.to(Cc.dtype)) * out_decay
    return (Y_diag + Y_off).reshape(Bsz, Tp, H, P)[:, :T], final_state

class Mamba2Block(nn.Module):
    def __init__(self, c: HybridConfig):
        super().__init__()
        self.d_inner, self.P, self.N, self.K, self.Q = c.expand * c.d_model, c.headdim, c.d_state, c.d_conv, c.chunk
        assert self.d_inner % self.P == 0
        self.H = self.d_inner // self.P
        self.conv_dim = self.d_inner + 2 * self.N
        self.in_proj = nn.Linear(c.d_model, 2 * self.d_inner + 2 * self.N + self.H, bias=False)   # z | x B C | dt
        self.conv1d = nn.Conv1d(self.conv_dim, self.conv_dim, self.K, groups=self.conv_dim, padding=self.K - 1)
        dt = torch.exp(torch.rand(self.H) * (math.log(0.1) - math.log(1e-3)) + math.log(1e-3)).clamp(min=1e-4)
        self.dt_bias = nn.Parameter(dt + torch.log(-torch.expm1(-dt)))            # softplus^-1(dt) with dt in [1e-3, 1e-1]
        self.A_log = nn.Parameter(torch.log(torch.empty(self.H).uniform_(1, 16)))
        self.D = nn.Parameter(torch.ones(self.H))
        self.norm = GatedRMSNorm(self.d_inner)
        self.out_proj = nn.Linear(self.d_inner, c.d_model, bias=False)

    def forward(self, x, valid=None, want_state=False):
        """Parallel (chunked) form for training and prompt prefill. valid: (B,T) bool, False = left-padding."""
        B, T, _ = x.shape
        z, xBC, dt = self.in_proj(x).split([self.d_inner, self.conv_dim, self.H], dim=-1)
        if valid is not None:
            xBC = xBC * valid[..., None].to(xBC.dtype)              # padding looks like the zero boundary of the conv
        xBC_t = xBC.transpose(1, 2)
        xBC = F.silu(self.conv1d(xBC_t)[..., :T]).transpose(1, 2)
        xs, Bm, Cm = xBC.split([self.d_inner, self.N, self.N], dim=-1)
        dtf = F.softplus(dt.float() + self.dt_bias)
        if valid is not None:
            dtf = dtf * valid[..., None].float()                    # dt = 0  =>  state passes through padding unchanged
        A = -torch.exp(self.A_log.float())
        X = xs.reshape(B, T, self.H, self.P)
        Y, state = ssd_chunked(X, dtf, A, Bm, Cm, self.Q)
        Y = Y + X * self.D.to(X.dtype)[None, None, :, None]
        out = self.out_proj(self.norm(Y.reshape(B, T, self.d_inner), z))
        if not want_state:
            return out, None
        conv_state = xBC_t[..., -(self.K - 1):]
        if conv_state.size(-1) < self.K - 1:
            conv_state = F.pad(conv_state, (self.K - 1 - conv_state.size(-1), 0))
        return out, (conv_state, state)

    def step(self, x, st):
        """Recurrent form for one token (B,1,D): constant memory, constant time."""
        conv_state, ssm = st
        z, xBC, dt = self.in_proj(x[:, 0]).split([self.d_inner, self.conv_dim, self.H], dim=-1)
        window = torch.cat([conv_state, xBC[..., None]], dim=-1)                    # (B, conv_dim, K)
        conv = (window.float() * self.conv1d.weight[:, 0, :].float()).sum(-1) + self.conv1d.bias.float()
        xs, Bm, Cm = F.silu(conv).split([self.d_inner, self.N, self.N], dim=-1)
        X = xs.view(-1, self.H, self.P)
        dtf = F.softplus(dt.float() + self.dt_bias)                                 # (B,H)
        dA = torch.exp(dtf * -torch.exp(self.A_log.float()))
        ssm = ssm * dA[..., None, None] + dtf[..., None, None] * X[..., None] * Bm[:, None, None, :]
        y = (ssm * Cm[:, None, None, :]).sum(-1) + self.D[None, :, None] * X        # (B,H,P)
        out = self.out_proj(self.norm(y.reshape(-1, 1, self.d_inner).to(z.dtype), z[:, None]))
        return out, (window[..., 1:], ssm)

def build_rope(head_dim, n_ctx, base):
    inv = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    freqs = torch.outer(torch.arange(n_ctx).float(), inv)
    return freqs.cos(), freqs.sin()

def apply_rope(x, cos, sin):                                 # x: (B, H, T, hd); cos/sin: (*, 1, T, hd/2)
    x1, x2 = x.float().chunk(2, dim=-1)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).type_as(x)

class Attention(nn.Module):
    """Grouped-query attention with RoPE (only 1 in 4 layers is attention in the default pattern)."""
    def __init__(self, c: HybridConfig):
        super().__init__()
        self.nh, self.nkv, self.hd = c.n_head, c.n_kv_head, c.d_model // c.n_head
        self.q = nn.Linear(c.d_model, self.nh * self.hd, bias=False)
        self.k = nn.Linear(c.d_model, self.nkv * self.hd, bias=False)
        self.v = nn.Linear(c.d_model, self.nkv * self.hd, bias=False)
        self.o = nn.Linear(self.nh * self.hd, c.d_model, bias=False)
    def forward(self, x, cos, sin, mask=None, cache=None):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        k = self.k(x).view(B, T, self.nkv, self.hd).transpose(1, 2)
        v = self.v(x).view(B, T, self.nkv, self.hd).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        new_cache = None
        if cache is not None:
            if cache[0] is not None:
                k, v = torch.cat([cache[0], k], 2), torch.cat([cache[1], v], 2)
            new_cache = (k, v)
        rep = self.nh // self.nkv
        if rep > 1:
            k, v = k.repeat_interleave(rep, 1), v.repeat_interleave(rep, 1)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, is_causal=(mask is None and cache is None))
        return self.o(y.transpose(1, 2).reshape(B, T, -1)), new_cache

class SwiGLU(nn.Module):
    def __init__(self, c: HybridConfig):
        super().__init__()
        self.gate = nn.Linear(c.d_model, c.n_ffn, bias=False)
        self.up = nn.Linear(c.d_model, c.n_ffn, bias=False)
        self.down = nn.Linear(c.n_ffn, c.d_model, bias=False)
    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class MambaLayer(nn.Module):
    kind = "M"
    def __init__(self, c):
        super().__init__()
        self.norm, self.mixer = RMSNorm(c.d_model), Mamba2Block(c)
    def forward(self, x, cos, sin, mask, valid, want_cache):
        h, st = self.mixer(self.norm(x), valid, want_cache)
        return x + h, st
    def step(self, x, cos, sin, key_mask, cache):
        h, st = self.mixer.step(self.norm(x), cache)
        return x + h, st

class AttnLayer(nn.Module):
    kind = "A"
    def __init__(self, c):
        super().__init__()
        self.n1, self.attn = RMSNorm(c.d_model), Attention(c)
        self.n2, self.mlp = RMSNorm(c.d_model), SwiGLU(c)
    def forward(self, x, cos, sin, mask, valid, want_cache):
        h, cache = self.attn(self.n1(x), cos, sin, mask, (None, None) if want_cache else None)
        x = x + h
        return x + self.mlp(self.n2(x)), cache
    def step(self, x, cos, sin, key_mask, cache):
        h, cache = self.attn(self.n1(x), cos, sin, key_mask, cache)
        x = x + h
        return x + self.mlp(self.n2(x)), cache

class LegalHybrid(nn.Module):
    def __init__(self, c: HybridConfig):
        super().__init__()
        assert set(c.pattern) <= {"M", "A"}
        self.c = c
        self.tok_emb = nn.Embedding(c.vocab_size, c.d_model)
        self.blocks = nn.ModuleList([MambaLayer(c) if ch == "M" else AttnLayer(c) for ch in c.pattern])
        self.norm = RMSNorm(c.d_model)
        self.head = nn.Linear(c.d_model, c.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight                                   # weight tying
        cos, sin = build_rope(c.d_model // c.n_head, c.n_ctx, c.rope_base)
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.grad_ckpt = False
        self.apply(self._init)
        for n, p in self.named_parameters():                                     # scaled residual init
            if n.endswith("out_proj.weight") or n.endswith("attn.o.weight") or n.endswith("mlp.down.weight"):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(len(c.pattern)))
    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None, valid=None, want_cache=False, last_only=False):
        """Training: model(x, y). Prompt prefill: model(idx, valid=left_pad_mask, want_cache=True, last_only=True)."""
        B, T = idx.shape
        x = self.tok_emb(idx)
        mask = None
        if valid is None:
            cos, sin = self.cos[:T][None, None], self.sin[:T][None, None]
        else:                                                                    # left-padded batch
            pos = (valid.long().cumsum(1) - 1).clamp(min=0)
            cos, sin = self.cos[pos][:, None], self.sin[pos][:, None]
            causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=idx.device))
            eye = torch.eye(T, dtype=torch.bool, device=idx.device)              # keeps pad rows finite
            mask = (causal[None] & (valid[:, None, :] | eye[None]))[:, None]
        caches = [] if want_cache else None
        for blk in self.blocks:
            if self.grad_ckpt and self.training:
                x = torch.utils.checkpoint.checkpoint(
                    lambda x_, c_, s_, b=blk: b(x_, c_, s_, None, None, False)[0], x, cos, sin, use_reentrant=False)
            else:
                x, c_new = blk(x, cos, sin, mask, valid, want_cache)
                if want_cache: caches.append(c_new)
        x = self.norm(x)
        if last_only: x = x[:, -1:]
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss, caches

    def step(self, tok, pos, key_mask, caches):
        """One decoding step. tok (B,1); pos (B,) = position of tok; key_mask (B,1,1,L_cached+1) bool."""
        x = self.tok_emb(tok)
        cos, sin = self.cos[pos][:, None, None, :], self.sin[pos][:, None, None, :]
        new = []
        for blk, cache in zip(self.blocks, caches):
            x, c_new = blk.step(x, cos, sin, key_mask, cache)
            new.append(c_new)
        return self.head(self.norm(x)), new

## 3. Load the Phase-1 checkpoint and tokenizer

In [4]:
for p in (CKPT_PATH, TOK_PATH):
    if not os.path.exists(p):
        print("MISSING:", p, "\nFiles under /kaggle/input:")
        for root, _, files in os.walk("/kaggle/input"):
            for f in files: print("  ", os.path.join(root, f))
        raise FileNotFoundError(p)
ck = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
p1_cfg = HybridConfig(**ck["cfg"])
tokenizer = ByteBPE.load(TOK_PATH)
EOS_ID = tokenizer.eos_id
assert p1_cfg.vocab_size == tokenizer.vocab_size, \
    f"tokenizer/checkpoint mismatch: model vocab {p1_cfg.vocab_size} vs tokenizer {tokenizer.vocab_size}. Use the bpe_merges.json this checkpoint was trained with."
assert WINDOW <= p1_cfg.n_ctx, f"WINDOW {WINDOW} exceeds the model's RoPE context {p1_cfg.n_ctx}"
print(f"checkpoint step {ck.get('step')} | pattern {p1_cfg.pattern} | d_model {p1_cfg.d_model} | vocab {p1_cfg.vocab_size} | n_ctx {p1_cfg.n_ctx}")
print("tokenizer merges md5:", hashlib.md5(open(TOK_PATH, "rb").read()).hexdigest()[:12])

def build_backbone():
    m = LegalHybrid(p1_cfg)
    m.load_state_dict(ck["model"])
    return m
_probe = build_backbone().eval()
print(f"Phase-1 backbone: {sum(p.numel() for p in _probe.parameters()) / 1e6:.1f}M parameters (loaded OK)")
del _probe

checkpoint step 1000 | pattern MMMAMMMAMMMAMMMAMMMA | d_model 768 | vocab 32000 | n_ctx 1280
tokenizer merges md5: 16ee00bde3c5
Phase-1 backbone: 111.0M parameters (loaded OK)


## 4. Phase-2 data: load, parse robustly, audit
The loader accepts the common JSON layouts (dict of texts or list of records; gold as lists or dicts). The audit **detects whether end offsets are inclusive or exclusive** from `text[start:end] == entity_text`, then converts everything to exclusive-end internally and converts back when writing the submission.

In [5]:
from huggingface_hub import HfApi, hf_hub_download, login
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("capstone-phase1-token")
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
_repo_files = HfApi().list_repo_files(HF_DATASET_REPO, repo_type="dataset", token=HF_TOKEN)

def fetch(*must_contain):
    hits = [f for f in _repo_files if all(s.lower() in f.lower() for s in must_contain)]
    if not hits:
        raise FileNotFoundError(f"{must_contain} not found. Repo has: {_repo_files[:60]}")
    return hf_hub_download(HF_DATASET_REPO, hits[0], repo_type="dataset", token=HF_TOKEN)
print([f for f in _repo_files if "phase_2" in f.lower()])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


['Phase_2_LNER/Test/fold3-text.json', 'Phase_2_LNER/Train/fold1-gold.json', 'Phase_2_LNER/Train/fold1-text.json', 'Phase_2_LNER/Train/fold2-gold.json', 'Phase_2_LNER/Train/fold2-text.json', 'Phase_2_LNER/Train/labels.txt']


In [6]:
P2 = {"f1_text": fetch("phase_2", "fold1-text"), "f1_gold": fetch("phase_2", "fold1-gold"),
      "f2_text": fetch("phase_2", "fold2-text"), "f2_gold": fetch("phase_2", "fold2-gold"),
      "labels": fetch("phase_2", "labels.txt"), "test": fetch("phase_2", "fold3-text")}

def load_json(p):
    with open(p, encoding="utf-8") as f: return json.load(f)

def parse_texts(obj):
    if isinstance(obj, dict):
        return {str(k): (v if isinstance(v, str) else v.get("text")) for k, v in obj.items()}
    return {str(r.get("id", r.get("doc_id", i))): r["text"] for i, r in enumerate(obj)}

def _first(d, keys):
    for k in keys:
        if k in d: return d[k]
    raise KeyError(f"none of {keys} in {list(d)}")

def parse_entity(e):
    if isinstance(e, (list, tuple)):
        return int(e[0]), int(e[1]), str(e[2]), (e[3] if len(e) > 3 else None)
    if isinstance(e, dict):
        s = _first(e, ["start", "start_char", "start_offset", "begin", "start_idx"])
        t = _first(e, ["end", "end_char", "end_offset", "stop", "end_idx"])
        l = _first(e, ["label", "type", "entity", "tag", "entity_type", "category"])
        x = next((e[k] for k in ("text", "entity_text", "span", "mention", "value") if k in e), None)
        return int(s), int(t), str(l), x
    raise ValueError(f"unrecognised entity format: {e!r}")

def parse_gold(obj):
    items = obj.items() if isinstance(obj, dict) else [(r.get("id", r.get("doc_id")), r) for r in obj]
    out = {}
    for k, v in items:
        if isinstance(v, dict):
            v = next((v[kk] for kk in v if "entit" in kk.lower() or "annot" in kk.lower()), [])
        out[str(k)] = [parse_entity(e) for e in v]
    return out

TEXT = {"fold1": parse_texts(load_json(P2["f1_text"])), "fold2": parse_texts(load_json(P2["f2_text"]))}
GOLD = {"fold1": parse_gold(load_json(P2["f1_gold"])), "fold2": parse_gold(load_json(P2["f2_gold"]))}
TEST_TEXT = parse_texts(load_json(P2["test"]))

raw_labels = [l.strip().split()[0] for l in open(P2["labels"], encoding="utf-8") if l.strip()]
LABELS = []
for l in raw_labels:
    base = re.sub(r"^[BI]-", "", l)
    if base != "O" and base not in LABELS: LABELS.append(base)
gold_labels = {e[2] for f in GOLD.values() for ents in f.values() for e in ents}
if gold_labels - set(LABELS):
    print("WARNING: labels in gold but not in labels.txt:", gold_labels - set(LABELS)); LABELS += sorted(gold_labels - set(LABELS))
TAGS = ["O"] + [f"{p}-{l}" for l in LABELS for p in "BI"]
TAG2ID = {t: i for i, t in enumerate(TAGS)}
K = len(TAGS)
print(f"{len(LABELS)} labels: {LABELS}  ->  {K} BIO tags")

# ---- offset convention: is `end` inclusive or exclusive? ----
exc = inc = tot = 0
for fold in GOLD:
    for did, ents in GOLD[fold].items():
        t = TEXT[fold].get(did)
        if t is None: continue
        for s, e, l, x in ents:
            if x is None: continue
            tot += 1; exc += (t[s:e] == x); inc += (t[s:e + 1] == x)
END_INCLUSIVE = bool(inc > exc)
print(f"offset convention check on {tot} entities: end-exclusive matches {exc} ({exc / max(tot, 1):.1%}), "
      f"end-inclusive matches {inc} ({inc / max(tot, 1):.1%}) -> END_INCLUSIVE = {END_INCLUSIVE}")
assert tot == 0 or max(exc, inc) / tot > 0.9, "offsets do not reproduce entity_text for >10% of gold: inspect samples before continuing"

GOLD_EXCL = {}                                   # internal convention: [start, end) with exclusive end
for fold in GOLD:
    GOLD_EXCL[fold] = {}
    for did, ents in GOLD[fold].items():
        n = len(TEXT[fold][did])
        conv = [(s, e + (1 if END_INCLUSIVE else 0), l) for s, e, l, x in ents]
        GOLD_EXCL[fold][did] = [(s, e, l) for s, e, l in conv if 0 <= s < e <= n and f"B-{l}" in TAG2ID]

# ---- audit ----
print("\nfold | docs | entities | median chars | max chars")
for fold in TEXT:
    ln = [len(t) for t in TEXT[fold].values()]
    print(f"{fold} | {len(ln)} | {sum(len(v) for v in GOLD_EXCL[fold].values())} | {int(np.median(ln))} | {max(ln)}")
ln = [len(t) for t in TEST_TEXT.values()]
print(f"fold3 (test) | {len(ln)} | - | {int(np.median(ln))} | {max(ln)}")
cnt = {f: Counter(l for v in GOLD_EXCL[f].values() for _, _, l in v) for f in GOLD_EXCL}
print("\nlabel counts:  label | fold1 | fold2")
for l in LABELS: print(f"  {l:10s} {cnt['fold1'].get(l, 0):7d} {cnt['fold2'].get(l, 0):7d}")
ov = sum(1 for f in GOLD_EXCL for v in GOLD_EXCL[f].values() for a in range(len(v)) for b in range(a + 1, len(v))
         if v[a][0] < v[b][1] and v[b][0] < v[a][1])
print(f"\noverlapping/nested gold pairs: {ov}  (flat BIO tagging cannot represent these)")
ex = next(((d, e) for d, v in GOLD["fold1"].items() for e in v[:1]), None)
print("sample gold entry:", ex)

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

12 labels: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']  ->  25 BIO tags
offset convention check on 8393 entities: end-exclusive matches 8393 (100.0%), end-inclusive matches 17 (0.2%) -> END_INCLUSIVE = False

fold | docs | entities | median chars | max chars
fold1 | 35 | 4120 | 26600 | 106947
fold2 | 35 | 4273 | 26730 | 114044
fold3 (test) | 35 | - | 28019 | 97103

label counts:  label | fold1 | fold2
  APP            207     200
  RESP           191     190
  A.COUNSEL       70     125
  R.COUNSEL       87      97
  JUDGE          123     122
  WIT             67     120
  AUTH           535     468
  COURT          455     558
  STAT           525     735
  PREC           624     559
  DATE           813     762
  CASENO         423     337

overlapping/nested gold pairs: 0  (flat BIO tagging cannot represent these)
sample gold entry: ('115651329', (137, 153, 'RESP', 'STATE OF HARYANA'))


## 5. Alignment: Phase-1 normalisation with an index map, and tokenisation with character offsets
Phase-1 models were trained on text after `normalize_text(..., keep_newlines=False)`: all whitespace collapsed, typographic quotes/dashes mapped, `…` expanded to `...`, zero-width characters removed. That changes character positions, so:

1. `normalize_with_map` reproduces `normalize_text` character by character and records, for every normalised character, the raw character range it came from. A hard assertion checks it equals `normalize_text` on every document.
2. Gold spans (raw offsets) are mapped into normalised coordinates for tagging; predicted spans are mapped back to **raw** offsets for the submission.
3. `encode_with_offsets` gives each BPE token its character span (trimming the leading space BPE tokens carry).
4. An **oracle test** turns the gold BIO tags straight back into spans and scores them: the ceiling any model can reach with this tokenisation.

In [7]:
def normalize_with_map(raw):
    """Equivalent to normalize_text(raw, keep_newlines=False) but also returns, for each output char,
    the raw range [rs, re) it came from."""
    out, rs, re_ = [], [], []
    for i, ch in enumerate(raw):
        for c in ch.translate(CHAR_MAP):                 # '' (deleted), ' ', '...', or the char itself
            out.append(" " if CTRL.match(c) else c); rs.append(i); re_.append(i + 1)
    t1 = "".join(out)
    res, rs2, re2, pos = [], [], [], 0
    for m in re.finditer(r"\s+", t1):
        a, b = m.span()
        res.extend(t1[pos:a]); rs2.extend(rs[pos:a]); re2.extend(re_[pos:a])
        res.append(" "); rs2.append(rs[a]); re2.append(re_[b - 1]); pos = b
    res.extend(t1[pos:]); rs2.extend(rs[pos:]); re2.extend(re_[pos:])
    lo, hi = 0, len(res)
    while lo < hi and res[lo] == " ": lo += 1
    while hi > lo and res[hi - 1] == " ": hi -= 1
    return "".join(res[lo:hi]), np.array(rs2[lo:hi], dtype=np.int64), np.array(re2[lo:hi], dtype=np.int64)

def encode_with_offsets(tok, text):
    """Same ids as tok.encode(text), plus the [start, end) character span of every token."""
    ids, offs = [], []
    for m in PRETOKEN_RE.finditer(text):
        chunk, cs = m.group(), m.start()
        tids = tok._encode_chunk(chunk)
        if len(chunk.encode("utf-8")) == len(chunk):                 # ASCII fast path
            b = 0
            for t in tids:
                nb = len(tok.vocab[t]); ids.append(t); offs.append((cs + b, cs + b + nb)); b += nb
        else:
            cob = []
            for ci, ch in enumerate(chunk): cob.extend([ci] * len(ch.encode("utf-8")))
            b = 0
            for t in tids:
                nb = len(tok.vocab[t]); ids.append(t); offs.append((cs + cob[b], cs + cob[b + nb - 1] + 1)); b += nb
    return ids, offs

STATS = Counter()

def prepare_doc(did, raw, ents_raw=None):
    norm, rs, re_ = normalize_with_map(raw)
    ids, offs = encode_with_offsets(tokenizer, norm)
    ts = np.array([o[0] for o in offs], dtype=np.int64); te = np.array([o[1] for o in offs], dtype=np.int64)
    for i in range(len(ts)):                                          # trim the leading space of a token
        while ts[i] < te[i] and norm[ts[i]] == " ": ts[i] += 1
    doc = dict(id=did, raw=raw, norm=norm, rs=rs, re=re_, ids=np.array(ids, dtype=np.int64), ts=ts, te=te)
    if ents_raw is not None:
        tags = np.zeros(len(ids), dtype=np.int64)
        ents = []
        for s, e, l in ents_raw:                                     # raw -> normalised coordinates
            ns = int(np.searchsorted(re_, s, side="right")); ne = int(np.searchsorted(rs, e, side="left"))
            STATS["gold"] += 1
            if ns >= ne: STATS["lost_in_normalisation"] += 1; continue
            if (int(rs[ns]), int(re_[ne - 1])) != (s, e): STATS["roundtrip_mismatch"] += 1
            ents.append((ns, ne, l))
        for ns, ne, l in sorted(ents, key=lambda x: -(x[1] - x[0])):  # longest first, so nested/overlapping shorter ones lose
            i0 = int(np.searchsorted(te, ns, side="right")); i1 = int(np.searchsorted(ts, ne, side="left")) - 1
            if i0 > i1: STATS["no_token"] += 1; continue
            if (tags[i0:i1 + 1] != 0).any(): STATS["overlap_dropped"] += 1; continue
            if ts[i0] != ns or te[i1] != ne: STATS["boundary_inside_token"] += 1
            tags[i0] = TAG2ID[f"B-{l}"]
            if i1 > i0: tags[i0 + 1:i1 + 1] = TAG2ID[f"I-{l}"]
        doc["tags"] = tags
    return doc

t0 = time.time()
DOCS = {f: [prepare_doc(d, TEXT[f][d], GOLD_EXCL[f][d]) for d in TEXT[f]] for f in TEXT}
TEST_DOCS = [prepare_doc(d, t) for d, t in TEST_TEXT.items()]
for f in DOCS:
    for d in DOCS[f]:                                                 # exactness of the normalisation map
        assert d["norm"] == normalize_text(d["raw"], False), f"normalisation map differs from normalize_text in doc {d['id']}"
print(f"prepared {sum(len(v) for v in DOCS.values())} labelled + {len(TEST_DOCS)} test docs in {time.time() - t0:.0f}s | normalisation map == normalize_text on all docs")
ntok = [len(d["ids"]) for f in DOCS for d in DOCS[f]]
print(f"tokens per doc: median {int(np.median(ntok))}, p90 {int(np.percentile(ntok, 90))}, max {max(ntok)} | tokenizer {sum(len(d['norm']) for f in DOCS for d in DOCS[f]) / sum(ntok):.2f} chars/token")
g = max(STATS["gold"], 1)
print(f"alignment report over {STATS['gold']} gold entities:")
for k in ("lost_in_normalisation", "roundtrip_mismatch", "no_token", "overlap_dropped", "boundary_inside_token"):
    print(f"   {k:24s} {STATS[k]:6d}  ({STATS[k] / g:.2%})")

prepared 70 labelled + 35 test docs in 5s | normalisation map == normalize_text on all docs
tokens per doc: median 5580, p90 10046, max 23860 | tokenizer 4.78 chars/token
alignment report over 8393 gold entities:
   lost_in_normalisation         0  (0.00%)
   roundtrip_mismatch            1  (0.01%)
   no_token                      0  (0.00%)
   overlap_dropped               0  (0.00%)
   boundary_inside_token       153  (1.82%)


In [8]:
# ---- decoding utilities (BIO -> spans -> raw offsets) ----
def make_trans_ok():
    ok = np.ones((K, K), dtype=bool)                       # ok[prev, next]
    for j in range(len(LABELS)):
        I = 2 + 2 * j; B = 1 + 2 * j
        for p in range(K):
            if p not in (B, I): ok[p, I] = False           # I-X only after B-X or I-X
    return ok
TRANS_OK = make_trans_ok()
START_OK = np.array([t == "O" or t.startswith("B-") for t in TAGS])
NEG = -1e9

def viterbi(logp):
    n = logp.shape[0]
    if n == 0: return np.zeros(0, dtype=np.int64)
    pen = np.where(TRANS_OK, 0.0, NEG).astype(np.float32)
    score = np.where(START_OK, logp[0], NEG).astype(np.float32)
    back = np.zeros((n, K), dtype=np.int16)
    for i in range(1, n):
        cand = score[:, None] + pen
        back[i] = cand.argmax(0)
        score = cand.max(0) + logp[i]
    best = int(score.argmax()); path = [best]
    for i in range(n - 1, 0, -1):
        best = int(back[i, best]); path.append(best)
    return np.array(path[::-1], dtype=np.int64)

def tags_to_token_spans(path):
    spans, cur = [], None
    for i, t in enumerate(path):
        tag = TAGS[int(t)]
        if tag == "O":
            if cur: spans.append(cur); cur = None
        elif tag.startswith("B-") or cur is None or cur[2] != tag[2:]:
            if cur: spans.append(cur)
            cur = [i, i, tag[2:]]
        else:
            cur[1] = i
    if cur: spans.append(cur)
    return spans

def decode_doc(doc, logp, method="viterbi"):
    path = viterbi(logp) if method == "viterbi" else logp.argmax(1)
    out = []
    for i0, i1, l in tags_to_token_spans(path):
        ns, ne = int(doc["ts"][i0]), int(doc["te"][i1])
        if ns >= ne: continue
        out.append((int(doc["rs"][ns]), int(doc["re"][ne - 1]), l))      # back to RAW offsets (exclusive end)
    return sorted(set(out))

def strict_scores(gold, pred):
    """gold/pred: {doc_id: [(start, end, label)]} with exclusive end. Exact start+end+label match."""
    G = {(d, s, e, l) for d, v in gold.items() for s, e, l in v}
    P = {(d, s, e, l) for d, v in pred.items() for s, e, l in v}
    per = {}
    for l in LABELS:
        g = {x for x in G if x[3] == l}; p = {x for x in P if x[3] == l}; tp = len(g & p)
        pr = tp / len(p) if p else 0.0; rc = tp / len(g) if g else 0.0
        per[l] = dict(p=pr, r=rc, f=(2 * pr * rc / (pr + rc) if pr + rc else 0.0), support=len(g), pred=len(p), tp=tp)
    present = [l for l in LABELS if per[l]["support"] > 0]
    tp = len(G & P); mp = tp / len(P) if P else 0.0; mr = tp / len(G) if G else 0.0
    return dict(macro_f1=float(np.mean([per[l]["f"] for l in present])) if present else 0.0,
                macro_p=float(np.mean([per[l]["p"] for l in present])) if present else 0.0,
                macro_r=float(np.mean([per[l]["r"] for l in present])) if present else 0.0,
                macro_f1_all_labels=float(np.mean([per[l]["f"] for l in LABELS])),
                micro_f1=(2 * mp * mr / (mp + mr) if mp + mr else 0.0), per_label=per)

def onehot_logp(doc):
    lp = np.full((len(doc["ids"]), K), -20.0, dtype=np.float32)
    lp[np.arange(len(lp)), doc["tags"]] = 0.0
    return lp

# ---- ORACLE: gold tags -> spans -> raw offsets. The ceiling for this tokenisation + decoding. ----
oracle = {}
for f in DOCS:
    pred = {d["id"]: decode_doc(d, onehot_logp(d), "viterbi") for d in DOCS[f]}
    oracle[f] = strict_scores(GOLD_EXCL[f], pred)
    print(f"oracle {f}: strict macro-F1 {oracle[f]['macro_f1']:.4f} | micro-F1 {oracle[f]['micro_f1']:.4f}   (ceiling from alignment/overlap losses)")

oracle fold1: strict macro-F1 0.9887 | micro-F1 0.9833   (ceiling from alignment/overlap losses)
oracle fold2: strict macro-F1 0.9848 | micro-F1 0.9801   (ceiling from alignment/overlap losses)


## 6. The model: Phase-1 Mamba hybrid + adaptation technique + BiGRU tagging head
* **Backbone:** the Phase-1 hybrid, run as a feature extractor (token embedding -> 20 layers -> final RMSNorm).
* **M1 full:** all weights trainable (backbone at a small LR, new head at a larger LR).
* **M2 LoRA:** every `nn.Linear` in the Mamba mixers (`in_proj`, `out_proj`) and attention layers (`q,k,v,o`, `gate,up,down`) gets a low-rank update `W x + (alpha/r) B A x`; the original weights are frozen.
* **M3 adapters:** a bottleneck adapter (down, GELU, up with zero-init, residual) after every layer; the backbone is frozen.
* **Head (all runs):** BiGRU (right context) -> dropout -> linear to the 25 BIO tags.

In [9]:
class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.t() @ self.B.t()) * self.scale

LORA_TARGETS = {"in_proj", "out_proj", "q", "k", "v", "o", "gate", "up", "down"}
def inject_lora(module, r, alpha, dropout):
    n = 0
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and name in LORA_TARGETS:
            setattr(module, name, LoRALinear(child, r, alpha, dropout)); n += 1
        else:
            n += inject_lora(child, r, alpha, dropout)
    return n

class Adapter(nn.Module):
    def __init__(self, d, r):
        super().__init__()
        self.down, self.up = nn.Linear(d, r), nn.Linear(r, d)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x):
        return self.up(F.gelu(self.down(x)))

class Phase2Model(nn.Module):
    def __init__(self, backbone, n_tags, exp):
        super().__init__()
        self.backbone, self.method = backbone, exp["method"]
        d = backbone.c.d_model
        self.adapters = None
        if self.method in ("lora", "adapter"):
            for p in backbone.parameters(): p.requires_grad = False
        if self.method == "lora":
            self.n_lora = inject_lora(backbone, exp["lora_r"], exp["lora_alpha"], exp["lora_dropout"])
        if self.method == "adapter":
            self.adapters = nn.ModuleList([Adapter(d, exp["adapter_dim"]) for _ in backbone.blocks])
        if CONTEXT_HEAD == "bigru":
            self.gru = nn.GRU(d, GRU_HIDDEN, batch_first=True, bidirectional=True); out_dim = 2 * GRU_HIDDEN
        else:
            self.gru, out_dim = None, d
        self.drop, self.cls = nn.Dropout(HEAD_DROPOUT), nn.Linear(out_dim, n_tags)

    def hidden(self, ids):
        m = self.backbone
        T = ids.size(1)
        x = m.tok_emb(ids)
        cos, sin = m.cos[:T][None, None], m.sin[:T][None, None]
        for i, blk in enumerate(m.blocks):
            if GRAD_CKPT and self.training:
                x = torch.utils.checkpoint.checkpoint(lambda x_, b=blk: b(x_, cos, sin, None, None, False)[0], x, use_reentrant=False)
            else:
                x, _ = blk(x, cos, sin, None, None, False)
            if self.adapters is not None:
                x = x + self.adapters[i](x)
        return m.norm(x)

    def forward(self, ids, lengths):
        h = self.hidden(ids)
        if self.gru is not None:
            pk = nn.utils.rnn.pack_padded_sequence(h, lengths.cpu(), batch_first=True, enforce_sorted=False)
            out, _ = self.gru(pk)
            h, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=ids.size(1))
        return self.cls(self.drop(h))

def build_model():
    return Phase2Model(build_backbone(), K, EXP).to(device)

def count_params(m):
    tot = sum(p.numel() for p in m.parameters()); tr = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return tot, tr

_m = build_model()
TOTAL_P, TRAIN_P = count_params(_m)
print(f"{EXPERIMENT}: total {TOTAL_P / 1e6:.2f}M params | trainable {TRAIN_P / 1e6:.2f}M ({100 * TRAIN_P / TOTAL_P:.1f}%)"
      + (f" | LoRA on {_m.n_lora} linear layers" if EXP["method"] == "lora" else ""))
del _m

M3_adapter: total 115.70M params | trainable 4.66M (4.0%)


## 7. Training and evaluation machinery
* Training windows: 1024 tokens, stride 512; windows without entities are kept with probability `NEG_WINDOW_KEEP` (re-drawn every epoch) to counter the dominant `O` class; rare entity tags are up-weighted (`sqrt_inv`, capped).
* Evaluation: overlapping windows -> averaged log-probabilities per token -> BIO-constrained Viterbi (or argmax) -> raw character spans -> **strict** match against raw gold.
* Epoch selection on the dev fold by strict macro-F1; the best epoch count is reused for the final retrain.

In [10]:
def window_starts(n, W, stride):
    if n <= W: return [0]
    s = list(range(0, n - W + 1, stride))
    if s[-1] + W < n: s.append(n - W)
    return s

def class_weights(docs):
    c = np.bincount(np.concatenate([d["tags"] for d in docs]), minlength=K).astype(np.float64)
    w = np.ones(K)
    if CLASS_WEIGHT == "sqrt_inv":
        ent = c[1:][c[1:] > 0]; ref = np.median(ent) if len(ent) else 1.0
        for t in range(1, K): w[t] = np.clip(np.sqrt(ref / max(c[t], 1.0)), 1.0, 5.0)
    return torch.tensor(w, dtype=torch.float32, device=device)

def collate(items, docs):
    L = max(l for _, _, l in items)
    ids = torch.full((len(items), L), EOS_ID, dtype=torch.long); lab = torch.full((len(items), L), -100, dtype=torch.long)
    lens = torch.zeros(len(items), dtype=torch.long)
    for b, (di, st, l) in enumerate(items):
        ids[b, :l] = torch.from_numpy(docs[di]["ids"][st:st + l]); lab[b, :l] = torch.from_numpy(docs[di]["tags"][st:st + l]); lens[b] = l
    return ids.to(device), lab.to(device), lens

@torch.no_grad()
def doc_logprobs(model, doc, bs=8):
    model.eval()
    n = len(doc["ids"]); acc = np.zeros((n, K), np.float32); wsum = np.zeros(n, np.float32)
    starts = window_starts(n, WINDOW, WINDOW - OVERLAP)
    for i in range(0, len(starts), bs):
        chunk = starts[i:i + bs]; lens = [min(WINDOW, n - s) for s in chunk]; L = max(lens)
        ids = torch.full((len(chunk), L), EOS_ID, dtype=torch.long)
        for b, (s, l) in enumerate(zip(chunk, lens)): ids[b, :l] = torch.from_numpy(doc["ids"][s:s + l])
        with torch.autocast(device, dtype=torch.float16, enabled=device == "cuda"):
            logits = model(ids.to(device), torch.tensor(lens))
        lp = F.log_softmax(logits.float(), -1).cpu().numpy()
        for b, (s, l) in enumerate(zip(chunk, lens)):
            w = np.minimum(np.arange(1, l + 1), np.arange(l, 0, -1)).astype(np.float32)   # triangular: trust window centres
            acc[s:s + l] += lp[b, :l] * w[:, None]; wsum[s:s + l] += w
    return acc / np.maximum(wsum, 1e-6)[:, None]

def evaluate(model, docs, gold, methods=("viterbi",)):
    t0 = time.time(); lps = {d["id"]: doc_logprobs(model, d) for d in docs}
    res = {}
    for m in methods:
        pred = {i: decode_doc(d, lps[d["id"]], m) for i, d in zip([d["id"] for d in docs], docs)}
        res[m] = (strict_scores(gold, pred), pred)
    return res, time.time() - t0

def run_training(train_docs, dev_docs, dev_gold, epochs, tag):
    model = build_model()
    cw = class_weights(train_docs)
    bb, hd = [], []
    for n, p in model.named_parameters():
        if p.requires_grad: (hd if n.startswith(("gru", "cls")) else bb).append((n, p))
    def groups(lst, lr):
        return [{"params": [p for n, p in lst if p.dim() >= 2], "lr": lr, "base_lr": lr, "weight_decay": WEIGHT_DECAY},
                {"params": [p for n, p in lst if p.dim() < 2], "lr": lr, "base_lr": lr, "weight_decay": 0.0}]
    opt = torch.optim.AdamW(groups(bb, EXP["lr_backbone"]) + groups(hd, EXP["lr_head"]), betas=(0.9, 0.98))
    scaler = torch.amp.GradScaler(device, enabled=device == "cuda")
    rng = random.Random(SEED)
    steps_per_epoch = max(1, len(build_items(train_docs, random.Random(SEED))) // (MICRO_BS * GRAD_ACCUM))
    total = steps_per_epoch * epochs; warm = max(1, int(WARMUP_FRAC * total)); step = 0
    trainable = {n for n, p in model.named_parameters() if p.requires_grad}
    best = dict(f1=-1.0, epoch=0, state=None); hist = []; t_start = time.time()
    if device == "cuda": torch.cuda.reset_peak_memory_stats()
    for ep in range(1, epochs + 1):
        model.train(); items = build_items(train_docs, rng); run, nb = 0.0, 0; t_ep = time.time()
        for i in range(0, len(items) - MICRO_BS * GRAD_ACCUM + 1, MICRO_BS * GRAD_ACCUM):
            f = (step + 1) / warm if step < warm else 0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm)))
            for g in opt.param_groups: g["lr"] = g["base_lr"] * f
            for a in range(GRAD_ACCUM):
                ids, lab, lens = collate(items[i + a * MICRO_BS: i + (a + 1) * MICRO_BS], train_docs)
                with torch.autocast(device, dtype=torch.float16, enabled=device == "cuda"):
                    logits = model(ids, lens)
                loss = F.cross_entropy(logits.float().view(-1, K), lab.view(-1), weight=cw, ignore_index=-100)
                if not math.isfinite(loss.item()): continue
                scaler.scale(loss / GRAD_ACCUM).backward(); run += loss.item(); nb += 1
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_([p for _, p in bb + hd], CLIP)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
        rec = dict(epoch=ep, train_loss=run / max(nb, 1), sec=time.time() - t_ep)
        if dev_docs is not None:
            res, sec_eval = evaluate(model, dev_docs, dev_gold, (DECODE_FOR_SELECTION,))
            sc = res[DECODE_FOR_SELECTION][0]; rec.update(dev_macro_f1=sc["macro_f1"], dev_micro_f1=sc["micro_f1"], eval_sec=sec_eval)
            if sc["macro_f1"] > best["f1"]:
                best = dict(f1=sc["macro_f1"], epoch=ep, state={n: v.detach().cpu().clone() for n, v in model.state_dict().items() if n in trainable})
            print(f"[{tag}] epoch {ep:2d}/{epochs} | loss {rec['train_loss']:.4f} | dev strict macro-F1 {sc['macro_f1']:.4f} (micro {sc['micro_f1']:.4f}) | {rec['sec']:.0f}s + eval {sec_eval:.0f}s")
        else:
            print(f"[{tag}] epoch {ep:2d}/{epochs} | loss {rec['train_loss']:.4f} | {rec['sec']:.0f}s")
        hist.append(rec)
        if (time.time() - t_start) / 60 > MAX_TRAIN_MINUTES and ep < epochs:
            print(f"[{tag}] time budget reached after epoch {ep}"); break
    if best["state"] is not None: model.load_state_dict(best["state"], strict=False)
    peak = torch.cuda.max_memory_allocated() / 1e9 if device == "cuda" else 0.0
    return model, dict(history=hist, best_epoch=best["epoch"] or len(hist), best_dev_f1=best["f1"],
                       train_minutes=(time.time() - t_start) / 60, peak_gpu_gb=peak)

def build_items(docs, rng):
    items = []
    for di, d in enumerate(docs):
        n = len(d["ids"])
        for st in window_starts(n, WINDOW, TRAIN_STRIDE):
            l = min(WINDOW, n - st)
            if (d["tags"][st:st + l] > 0).any() or rng.random() < NEG_WINDOW_KEEP: items.append((di, st, l))
    rng.shuffle(items)
    return items

## 8. Phase A: train on fold1, select the epoch on fold2 (strict macro-F1)

In [11]:
train_docs = DOCS["fold1"] if DEV_FOLD == "fold2" else DOCS["fold2"]
dev_docs = DOCS[DEV_FOLD]; dev_gold = GOLD_EXCL[DEV_FOLD]
print(f"train docs {len(train_docs)} | dev docs {len(dev_docs)} ({DEV_FOLD}) | training windows/epoch ~{len(build_items(train_docs, random.Random(0)))}")
model, infoA = run_training(train_docs, dev_docs, dev_gold, EPOCHS, f"{EXPERIMENT}/A")
resA, sec_infer = evaluate(model, dev_docs, dev_gold, ("viterbi", "argmax"))
scV, predV = resA["viterbi"]; scA = resA["argmax"][0]
print(f"\nbest epoch {infoA['best_epoch']} | DEV strict macro-F1: viterbi {scV['macro_f1']:.4f} | argmax {scA['macro_f1']:.4f} | "
      f"micro-F1 (viterbi) {scV['micro_f1']:.4f} | macro-F1 over all 12 labels {scV['macro_f1_all_labels']:.4f}")
print(f"oracle ceiling on {DEV_FOLD}: {oracle[DEV_FOLD]['macro_f1']:.4f} | dev inference {sec_infer:.0f}s for {len(dev_docs)} docs")
print("\nlabel      |  P     R     F1   | support")
for l in LABELS:
    r = scV["per_label"][l]; print(f"  {l:10s} | {r['p']:.3f} {r['r']:.3f} {r['f']:.3f} | {r['support']}")

train docs 35 | dev docs 35 (fold2) | training windows/epoch ~429
[M3_adapter/A] epoch  1/10 | loss 0.7377 | dev strict macro-F1 0.1212 (micro 0.2563) | 56s + eval 18s
[M3_adapter/A] epoch  2/10 | loss 0.1315 | dev strict macro-F1 0.2682 (micro 0.4341) | 62s + eval 18s
[M3_adapter/A] epoch  3/10 | loss 0.0784 | dev strict macro-F1 0.3742 (micro 0.5422) | 62s + eval 18s
[M3_adapter/A] epoch  4/10 | loss 0.0533 | dev strict macro-F1 0.4470 (micro 0.5742) | 62s + eval 18s
[M3_adapter/A] epoch  5/10 | loss 0.0358 | dev strict macro-F1 0.4759 (micro 0.6069) | 62s + eval 18s
[M3_adapter/A] epoch  6/10 | loss 0.0223 | dev strict macro-F1 0.4450 (micro 0.5754) | 62s + eval 18s
[M3_adapter/A] epoch  7/10 | loss 0.0139 | dev strict macro-F1 0.4478 (micro 0.5909) | 62s + eval 18s
[M3_adapter/A] epoch  8/10 | loss 0.0091 | dev strict macro-F1 0.4656 (micro 0.5971) | 62s + eval 18s
[M3_adapter/A] epoch  9/10 | loss 0.0068 | dev strict macro-F1 0.4526 (micro 0.5880) | 62s + eval 18s
[M3_adapter/A] e

### Error analysis on the dev fold
Splits every wrong prediction into *boundary errors* (same label, overlapping gold span, wrong offsets), *label confusions* (overlaps a gold span of another label) and *spurious* spans; missed gold spans are counted too. The strict metric makes boundary errors cost as much as complete misses.

In [14]:
def error_analysis(gold, pred, docs):
    G = {d: list(v) for d, v in gold.items()}
    boundary = confusion = spurious = missed = 0; conf_pairs = Counter(); ex = []
    for d, preds in pred.items():
        gs = G.get(d, []); gset = set(gs)
        for p in preds:
            if p in gset: continue
            ov = [g for g in gs if g[0] < p[1] and p[0] < g[1]]
            if any(g[2] == p[2] for g in ov): boundary += 1; kind = "boundary"
            elif ov: confusion += 1; conf_pairs[(ov[0][2], p[2])] += 1; kind = "label"
            else: spurious += 1; kind = "spurious"
            if len(ex) < 12: ex.append((kind, d, p, ov[:1]))
        pset = set(preds)
        missed += sum(1 for g in gs if g not in pset and not any(q[0] < g[1] and g[0] < q[1] for q in preds))
    return dict(boundary=boundary, label_confusion=confusion, spurious=spurious, missed_entirely=missed,
                top_confusions=conf_pairs.most_common(8)), ex

ea, examples = error_analysis(dev_gold, predV, dev_docs)
tot_wrong = ea["boundary"] + ea["label_confusion"] + ea["spurious"]
print({k: v for k, v in ea.items() if k != "top_confusions"}, f"(wrong predictions: {tot_wrong})")
print("top label confusions (gold -> predicted):", ea["top_confusions"])
by_id = {d["id"]: d for d in dev_docs}
for kind, d, p, ov in examples[:8]:
    raw = by_id[d]["raw"]
    print(f"  [{kind:8s}] pred {p[2]:9s} '{raw[p[0]:p[1]][:60]}'" + (f"   gold {ov[0][2]} '{raw[ov[0][0]:ov[0][1]][:60]}'" if ov else ""))

{'boundary': 527, 'label_confusion': 369, 'spurious': 693, 'missed_entirely': 810} (wrong predictions: 1589)
top label confusions (gold -> predicted): [(('A.COUNSEL', 'R.COUNSEL'), 88), (('WIT', 'APP'), 33), (('WIT', 'RESP'), 27), (('PREC', 'COURT'), 21), (('RESP', 'APP'), 18), (('WIT', 'PREC'), 16), (('COURT', 'PREC'), 14), (('APP', 'RESP'), 13)]
  [label   ] pred PREC      'Kolkata Municipal Corporation Vs. Smt. Kajari'   gold APP 'Kolkata Municipal Corporation'
  [label   ] pred R.COUNSEL 'Alok Kr. Ghosh'   gold A.COUNSEL 'Ghosh'
  [spurious] pred R.COUNSEL 'S.K. Debnath'
  [label   ] pred DATE      '18.12.2020'   gold CASENO '18.12.2020'
  [label   ] pred DATE      '22.12.2020'   gold CASENO '22.12.2020'
  [boundary] pred STAT      'Constitution of'   gold STAT 'Constitution of India'
  [label   ] pred RESP      'Kolkata Municipal Corporation'   gold APP 'Kolkata Municipal Corporation'
  [label   ] pred RESP      'Municipal Corporation'   gold APP 'Kolkata Municipal Corporation'


In [15]:
FINAL_RETRAIN = True         

@torch.no_grad()
def doc_logprobs(model, doc, bs=8):
    model.eval()
    n = len(doc["ids"])
    if n == 0:                                   # empty document: nothing to tag
        return np.zeros((0, K), np.float32)
    acc = np.zeros((n, K), np.float32); wsum = np.zeros(n, np.float32)
    starts = window_starts(n, WINDOW, WINDOW - OVERLAP)
    for i in range(0, len(starts), bs):
        chunk = starts[i:i + bs]; lens = [min(WINDOW, n - s) for s in chunk]; L = max(lens)
        ids = torch.full((len(chunk), L), EOS_ID, dtype=torch.long)
        for b, (s, l) in enumerate(zip(chunk, lens)): ids[b, :l] = torch.from_numpy(doc["ids"][s:s + l])
        with torch.autocast(device, dtype=torch.float16, enabled=device == "cuda"):
            logits = model(ids.to(device), torch.tensor(lens))
        lp = F.log_softmax(logits.float(), -1).cpu().numpy()
        for b, (s, l) in enumerate(zip(chunk, lens)):
            w = np.minimum(np.arange(1, l + 1), np.arange(l, 0, -1)).astype(np.float32)
            acc[s:s + l] += lp[b, :l] * w[:, None]; wsum[s:s + l] += w
    return acc / np.maximum(wsum, 1e-6)[:, None]

print([(d["id"], len(d["raw"]), repr(d["raw"][:40])) for d in TEST_DOCS if len(d["ids"]) == 0])

[('1305957', 0, "''")]


## 9. Phase B: final model on fold1+fold2, predict fold3, write and validate the submission

In [16]:
if FINAL_RETRAIN:
    del model
    if device == "cuda": torch.cuda.empty_cache()
    final_epochs = max(3, infoA["best_epoch"])
    model, infoB = run_training(DOCS["fold1"] + DOCS["fold2"], None, None, final_epochs, f"{EXPERIMENT}/B")
else:
    infoB = dict(note="used the phase-A model (fold1 only)", best_epoch=infoA["best_epoch"])

t0 = time.time()
test_pred = {d["id"]: decode_doc(d, doc_logprobs(model, d), "viterbi") for d in TEST_DOCS}
infer_sec = time.time() - t0

def to_output(pred, docs):
    out = {}
    by = {d["id"]: d for d in docs}
    for did in by:
        rows = []
        for s, e, l in sorted(pred.get(did, [])):
            row = [int(s), int(e - 1 if END_INCLUSIVE else e), l]
            if INCLUDE_ENTITY_TEXT: row.append(by[did]["raw"][s:e])
            rows.append(row)
        out[did] = rows
    return out

submission = to_output(test_pred, TEST_DOCS)
# ---- validation of the file the scorer will read ----
assert set(submission) == set(TEST_TEXT), "document IDs differ from fold3-text.json"
bad = 0
for did, rows in submission.items():
    n = len(TEST_TEXT[did])
    for r in rows:
        s, e, l = r[0], r[1], r[2]
        e_x = e + 1 if END_INCLUSIVE else e
        if not (isinstance(s, int) and isinstance(e, int) and l in LABELS and 0 <= s < e_x <= n): bad += 1
assert bad == 0, f"{bad} malformed predictions"
n_pred = sum(len(v) for v in submission.values())
print(f"test: {len(submission)} docs | {n_pred} predicted entities | {n_pred / len(submission):.1f} per doc | END_INCLUSIVE={END_INCLUSIVE} | inference {infer_sec:.0f}s")
print("predicted label distribution:", dict(Counter(r[2] for v in submission.values() for r in v).most_common()))
d0 = next(iter(submission))
for r in submission[d0][:5]:
    e_x = r[1] + 1 if END_INCLUSIVE else r[1]
    print(f"  doc {d0}: {r[:3]} -> '{TEST_TEXT[d0][r[0]:e_x][:70]}'")

PRED_PATH = f"{EXP_DIR}/prediction.json"; ZIP_PATH = f"{EXP_DIR}/submission_{EXPERIMENT}.zip"
json.dump(submission, open(PRED_PATH, "w"))
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z: z.write(PRED_PATH, arcname="prediction.json")
with zipfile.ZipFile(ZIP_PATH) as z: print("submission archive:", ZIP_PATH, "->", z.namelist())

[M3_adapter/B] epoch  1/5 | loss 0.4626 | 129s
[M3_adapter/B] epoch  2/5 | loss 0.0881 | 128s
[M3_adapter/B] epoch  3/5 | loss 0.0587 | 129s
[M3_adapter/B] epoch  4/5 | loss 0.0376 | 126s
[M3_adapter/B] epoch  5/5 | loss 0.0270 | 126s
test: 35 docs | 4003 predicted entities | 114.4 per doc | END_INCLUSIVE=False | inference 20s
predicted label distribution: {'STAT': 872, 'DATE': 737, 'PREC': 621, 'COURT': 558, 'AUTH': 316, 'CASENO': 306, 'APP': 149, 'A.COUNSEL': 106, 'JUDGE': 105, 'R.COUNSEL': 86, 'RESP': 80, 'WIT': 67}
  doc 1128835: [70, 89, 'CASENO'] -> 'CIVIL APPEAL NO.658'
  doc 1128835: [163, 183, 'APP'] -> 'ESSCO FABS PVT. LTD.'
  doc 1128835: [213, 229, 'RESP'] -> 'STATE OF HARYANA'
  doc 1128835: [258, 277, 'CASENO'] -> 'CIVIL APPEAL NO.658'
  doc 1128835: [355, 371, 'APP'] -> 'PANIPAT TEACHERS'
submission archive: /kaggle/working/phase2_lner/M3_adapter/submission_M3_adapter.zip -> ['prediction.json']


## 10. Save the experiment record and compare runs

In [39]:
result = dict(experiment=EXPERIMENT, description=EXP["desc"], method=EXP["method"],
              total_params=TOTAL_P, trainable_params=TRAIN_P, trainable_pct=100 * TRAIN_P / TOTAL_P,
              phaseA=dict(best_epoch=infoA["best_epoch"], dev_fold=DEV_FOLD, dev_macro_f1_viterbi=scV["macro_f1"],
                          dev_macro_f1_argmax=scA["macro_f1"], dev_micro_f1=scV["micro_f1"],
                          dev_macro_f1_all_labels=scV["macro_f1_all_labels"], per_label=scV["per_label"],
                          train_minutes=infoA["train_minutes"], peak_gpu_gb=infoA["peak_gpu_gb"], history=infoA["history"],
                          errors={k: v for k, v in ea.items() if k != "top_confusions"}, top_confusions=ea["top_confusions"]),
              phaseB=dict(infoB, test_inference_sec=infer_sec, n_pred=n_pred),
              oracle_dev_macro_f1=oracle[DEV_FOLD]["macro_f1"], alignment=dict(STATS), end_inclusive=END_INCLUSIVE,
              config=dict(window=WINDOW, overlap=OVERLAP, epochs=EPOCHS, context_head=CONTEXT_HEAD, class_weight=CLASS_WEIGHT,
                          neg_window_keep=NEG_WINDOW_KEEP, lr_backbone=EXP["lr_backbone"], lr_head=EXP["lr_head"]))
json.dump(result, open(f"{OUTPUT_DIR}/results_{EXPERIMENT}.json", "w"), indent=1, default=float)

import glob
rows = []
for p in sorted(glob.glob(f"{OUTPUT_DIR}/results_*.json")):
    r = json.load(open(p)); a = r["phaseA"]
    rows.append((r["experiment"], r["method"], r["trainable_params"] / 1e6, a["dev_macro_f1_viterbi"], a["dev_macro_f1_argmax"],
                 a["errors"]["boundary"], a["train_minutes"], a["peak_gpu_gb"]))
print("\nexperiment    | method  | trainable M | dev macro-F1 (viterbi) | argmax | boundary errs | train min | peak GB")
for r in rows: print(f"{r[0]:13s} | {r[1]:7s} | {r[2]:11.2f} | {r[3]:22.4f} | {r[4]:.4f} | {r[5]:13d} | {r[6]:9.1f} | {r[7]:.1f}")
print("\nUpload", ZIP_PATH, "to Codabench (Phase 2).")


experiment    | method  | trainable M | dev macro-F1 (viterbi) | argmax | boundary errs | train min | peak GB
M1_full       | full    |      113.72 |                 0.4695 | 0.4055 |           525 |      13.3 | 7.5
M2_lora       | lora    |        5.27 |                 0.4803 | 0.4098 |           562 |      13.9 | 6.5

Upload /kaggle/working/phase2_lner/M2_lora/submission_M2_lora.zip to Codabench (Phase 2).


## Notes for the report
* All three runs share data, windows, head, decoding and epoch budget, so differences come from the adaptation technique alone. Report trainable parameters, training minutes, peak GPU memory and per-label F1 for each (all saved in `results_*.json`).
* `oracle` is the best strict F1 any model could reach with this tokenisation, normalisation map and decoding: use it to separate modelling errors from alignment losses.
* Ablations that need no new technique (do **not** count toward the six): `CONTEXT_HEAD="none"` (is the BiGRU helping?), `CLASS_WEIGHT="none"`, `NEG_WINDOW_KEEP`, `WINDOW`/`OVERLAP`, `DECODE_FOR_SELECTION="argmax"`.
* **LORMA is mandatory** for the six submissions and is not part of this notebook; it belongs with the other Phase-1 models (decoder-only is the natural place). It needs its definition from reference [12].
* The six submissions must involve **different Phase-1 models**, so these three Mamba runs are only part of the final set.